# 02 · Preprocessing dan Pemahaman Data
**Bab III Subbab 3.2 dan 3.3.** Validasi tipe dan rentang, pemeriksaan satuan `timestamp`, penanganan data kosong,
normalisasi teks, penghapusan duplikat, fitur panjang, dan usia ulasan. Setelah itu statistik deskriptif untuk Bab IV.

**Keluaran:** `reviews_clean.parquet`, `preprocess_report.json`, `describe.json`, `tables/eda_*.csv`, `figures/eda_*.png`

> **Google Colab.** Jalankan sekali per sesi sebelum sel setup:
> ```
> !git clone -b main https://github.com/Vito28/dl-paper.git
> %cd dl-paper/notebooks
> !pip install -q -r ../requirements.txt
> ```
> Sel setup otomatis me-mount Google Drive (izinkan akses saat diminta). **Semua data dan hasil, termasuk
> unduhan mentah, disimpan di `MyDrive/dl-paper`**, bukan di instance. Jika Drive gagal di-mount atau
> terputus, notebook berhenti dengan error. Jalankan sel terakhir (`finish()`) agar semua tulisan tersinkron.

In [ ]:
# Setup: path proyek dan konfigurasi bersama (ubah konfigurasi di notebooks/settings.py)
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "notebooks")]
from settings import cfg, SMOKE, finish
import numpy as np, pandas as pd
pd.set_option("display.width", 160)
print("kategori:", cfg.category, "| output:", cfg.out_dir, "| SMOKE:", SMOKE)

In [ ]:
from hr import data, plots
import matplotlib.pyplot as plt

## 1. Baca data mentah
Jika berkas terlalu besar untuk RAM, turunkan `product_sample_rate` di `settings.py`. Sampel diambil per produk, sehingga daftar ulasan setiap produk tetap utuh.

In [ ]:
raw = data.load_reviews(cfg)
print(len(raw), "baris")
raw.head(3)

## 2. Preprocessing
Langkah 1–6 pada Subbab 3.3 dijalankan berurutan. Laporan menyimpan berapa baris yang dibuang pada setiap langkah.

In [ ]:
df, rep = data.preprocess(raw, cfg)
del raw
cfg.path("preprocess_report.json").write_text(json.dumps(rep, indent=2, default=str))
pd.Series({k: v for k, v in rep.items() if k != "timestamp_check"})

In [ ]:
rep["timestamp_check"]

## 3. Statistik deskriptif

In [ ]:
desc = data.describe(df)
cfg.path("describe.json").write_text(json.dumps(desc, indent=2))
pd.Series(desc)

### Sebaran helpful votes, usia ulasan, jumlah ulasan per produk, dan panjang ulasan

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6.5))
plots.vote_distribution(df["helpful_vote"].to_numpy(), ax=axes[0, 0])
plots.simple_hist(df["age_days"] / 365.25, "usia ulasan (tahun)", "Sebaran usia ulasan", ax=axes[0, 1])
plots.simple_hist(np.log10(df.groupby("parent_asin").size()), "log10 jumlah ulasan per produk",
                  "Ulasan per produk", log_y=True, ax=axes[1, 0])
plots.simple_hist(np.log10(df["word_count"] + 1), "log10 jumlah kata", "Panjang isi ulasan", ax=axes[1, 1])
fig.tight_layout()
plots.save(fig, cfg.path("figures", "eda_distribusi.png"))

### Usia ulasan dan helpful votes
Tabel ini memberi gambaran awal untuk RM3 sebelum model Negative Binomial di notebook 03.

In [ ]:
q = pd.qcut(df["age_days"], 5, labels=["Q1 (baru)", "Q2", "Q3", "Q4", "Q5 (lama)"])
age_tab = df.groupby(q, observed=True)["helpful_vote"].agg(
    ulasan="size", rata_rata="mean", median="median", proporsi_nol=lambda v: (v == 0).mean()).reset_index()
age_tab.to_csv(cfg.path("tables", "eda_usia_vs_vote.csv"), index=False)
age_tab

## 4. Panjang token
Proporsi ulasan yang terpotong oleh batas 512 token (dihitung dari sampel).

In [ ]:
from hr.embed import token_length_stats
tok_stats = token_length_stats(df, cfg.encoders["blair"], cfg.max_len, n=50_000 if not SMOKE else 500)
cfg.path("tables", "eda_token.json").write_text(json.dumps(tok_stats, indent=2))
tok_stats

## 5. Simpan data bersih

In [ ]:
data.save_parquet(df, cfg.path("reviews_clean.parquet"))
print(df.shape)

## Simpan ke Google Drive
Memastikan semua hasil notebook ini sudah tersinkron ke Drive sebelum runtime ditutup.

In [ ]:
finish()